In [38]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display, clear_output

In [39]:
raw_df = px.data.gapminder()

columns = [
    "country",
    "continent",
    "year",
    "lifeExp",
    "pop",
    "gdpPercap"
]

df = raw_df[columns].copy()
df = df.drop_duplicates()

numeric_columns = [
    "year",
    "lifeExp",
    "pop",
    "gdpPercap"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df = df.dropna(subset=columns)

df = df[
    (df["lifeExp"] > 0)
    & (df["pop"] > 0)
    & (df["gdpPercap"] > 0)
]

df["year"] = df["year"].astype(int)

df = (
    df.sort_values(["country", "year"])
      .reset_index(drop=True)
)

df.head()

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


In [45]:
years = sorted(df["year"].unique())
continents = sorted(df["continent"].unique())

continent_colors = {
    "Africa": "#636EFA",
    "Americas": "#EF553B",
    "Asia": "#00CC96",
    "Europe": "#AB63FA",
    "Oceania": "#FFA15A"
}

continent_widget = widgets.RadioButtons(
    options=["All"] + continents,
    value="All",
    description="",
    layout=widgets.Layout(
        width="170px"
    )
)

year_widget = widgets.SelectionRangeSlider(
    options=years,
    index=(0, len(years) - 1),
    description="",
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)

scale_widget = widgets.RadioButtons(
    options=[
        ("Logarithmic", "log"),
        ("Linear", "linear")
    ],
    value="log",
    description="",
    layout=widgets.Layout(width="180px")
)

In [46]:
summary_output = widgets.Output()

scatter_output = widgets.Output()
line_output = widgets.Output()
bar_output = widgets.Output()
histogram_output = widgets.Output()

In [47]:
def update_dashboard(change=None):
    selected_continent = continent_widget.value
    start_year, end_year = year_widget.value
    selected_scale = scale_widget.value

    filtered = df[
        df["year"].between(start_year, end_year)
    ].copy()

    if selected_continent != "All":
        filtered = filtered[
            filtered["continent"] == selected_continent
        ]

    current = filtered[
        filtered["year"] == end_year
    ].copy()

    if current.empty:
        with summary_output:
            clear_output(wait=True)
            print("No data are available for this selection.")

        for output in [
            scatter_output,
            line_output,
            bar_output,
            histogram_output
        ]:
            with output:
                clear_output(wait=True)
                print("No data available.")

        return

    # summary inddicators

    country_count = current["country"].nunique()
    total_population = current["pop"].sum()

    weighted_life = (
        current["lifeExp"].mul(current["pop"]).sum()
        / total_population
    )

    if total_population >= 1_000_000_000:
        population_text = (
            f"{total_population / 1_000_000_000:.2f} billion"
        )
    else:
        population_text = (
            f"{total_population / 1_000_000:.1f} million"
        )

    with summary_output:
        clear_output(wait=True)

        summary_html = widgets.HTML(
            value=f"""
            <div style="
                display:flex;
                gap:20px;
                flex-wrap:wrap;
                margin:10px 0 15px 0;
            ">
                <div style="
                    flex:1;
                    min-width:180px;
                    padding:15px;
                    background:white;
                    border:1px solid #dddddd;
                    border-radius:8px;
                    text-align:center;
                ">
                    <div style="color:#666666;">Countries</div>
                    <div style="font-size:25px;font-weight:bold;">
                        {country_count:,}
                    </div>
                </div>

                <div style="
                    flex:1;
                    min-width:180px;
                    padding:15px;
                    background:white;
                    border:1px solid #dddddd;
                    border-radius:8px;
                    text-align:center;
                ">
                    <div style="color:#666666;">Population</div>
                    <div style="font-size:25px;font-weight:bold;">
                        {population_text}
                    </div>
                </div>

                <div style="
                    flex:1;
                    min-width:180px;
                    padding:15px;
                    background:white;
                    border:1px solid #dddddd;
                    border-radius:8px;
                    text-align:center;
                ">
                    <div style="color:#666666;">
                        Weighted life expectancy
                    </div>
                    <div style="font-size:25px;font-weight:bold;">
                        {weighted_life:.1f} years
                    </div>
                </div>
            </div>
            """
        )

        display(summary_html)

    # scatter plot

    scatter = px.scatter(
        current,
        x="gdpPercap",
        y="lifeExp",
        size="pop",
        color="continent",
        hover_name="country",
        size_max=45,
        log_x=(selected_scale == "log"),
        color_discrete_map=continent_colors,
        title=f"GDP and Life Expectancy, {end_year}",
        labels={
            "gdpPercap": "GDP per capita",
            "lifeExp": "Life expectancy",
            "pop": "Population",
            "continent": "Continent"
        }
    )

    # line chart

    trend = filtered.copy()

    trend["weighted_total"] = (
        trend["lifeExp"] * trend["pop"]
    )

    trend = (
        trend.groupby(
            ["continent", "year"],
            as_index=False
        )
        .agg(
            weighted_total=("weighted_total", "sum"),
            total_population=("pop", "sum")
        )
    )

    trend["weighted_life_expectancy"] = (
        trend["weighted_total"]
        / trend["total_population"]
    )

    line = px.line(
        trend,
        x="year",
        y="weighted_life_expectancy",
        color="continent",
        markers=True,
        color_discrete_map=continent_colors,
        title=(
            "Population-Weighted Life Expectancy, "
            f"{start_year}–{end_year}"
        ),
        labels={
            "year": "Year",
            "weighted_life_expectancy":
                "Life expectancy",
            "continent": "Continent"
        }
    )

    line.update_layout(hovermode="x unified")

    # bar chart

    top_ten = (
        current.nlargest(10, "pop")
               .sort_values("pop")
    )

    bar = px.bar(
        top_ten,
        x="pop",
        y="country",
        color="continent",
        orientation="h",
        text_auto=".3s",
        color_discrete_map=continent_colors,
        title=f"Ten Most Populous Countries, {end_year}",
        labels={
            "pop": "Population",
            "country": "",
            "continent": "Continent"
        }
    )

    bar.update_layout(showlegend=False)

    # histogram

    histogram = px.histogram(
        current,
        x="lifeExp",
        color="continent",
        nbins=15,
        barmode="overlay",
        opacity=0.7,
        color_discrete_map=continent_colors,
        title=(
            "Distribution of Life Expectancy, "
            f"{end_year}"
        ),
        labels={
            "lifeExp": "Life expectancy",
            "count": "Number of countries",
            "continent": "Continent"
        }
    )

    histogram.update_layout(bargap=0.05)

    # Consistent size and appearance
    for figure in [scatter, line, bar, histogram]:
        figure.update_layout(
            template="plotly_white",
            height=420,
            margin={
                "l": 50,
                "r": 20,
                "t": 60,
                "b": 45
            }
        )

    # Display each graph in its output area
    with scatter_output:
        clear_output(wait=True)
        scatter.show()

    with line_output:
        clear_output(wait=True)
        line.show()

    with bar_output:
        clear_output(wait=True)
        bar.show()

    with histogram_output:
        clear_output(wait=True)
        histogram.show()

In [48]:
continent_widget.observe(
    update_dashboard,
    names="value"
)

year_widget.observe(
    update_dashboard,
    names="value"
)

scale_widget.observe(
    update_dashboard,
    names="value"
)

In [49]:
title = widgets.HTML(
    value="""
    <div style="text-align:center;">
        <h1 style="color:#17324D;margin-bottom:5px;">
            Global Development Explorer
        </h1>
        <p style="color:#526579;">
            Explore global health, wealth, and population
            from 1952 through 2007.
        </p>
    </div>
    """
)

# Create separate labels above each widget.
continent_control = widgets.VBox(
    [
        widgets.HTML("<b>Continent</b>"),
        continent_widget
    ],
    layout=widgets.Layout(
        width="250px",
        padding="5px"
    )
)

year_control = widgets.VBox(
    [
        widgets.HTML("<b>Year range</b>"),
        year_widget
    ],
    layout=widgets.Layout(
        width="470px",
        padding="5px"
    )
)

scale_control = widgets.VBox(
    [
        widgets.HTML("<b>GDP axis</b>"),
        scale_widget
    ],
    layout=widgets.Layout(
        width="200px",
        padding="5px"
    )
)

controls = widgets.HBox(
    [
        continent_control,
        year_control,
        scale_control
    ],
    layout=widgets.Layout(
        display="flex",
        flex_flow="row wrap",
        justify_content="space-around",
        align_items="flex-start",
        width="100%",
        border="1px solid #dddddd",
        padding="12px",
        margin="5px 0 10px 0"
    )
)

graph_grid = widgets.GridBox(
    children=[
        scatter_output,
        line_output,
        bar_output,
        histogram_output
    ],
    layout=widgets.Layout(
        grid_template_columns="repeat(2, minmax(400px, 1fr))",
        grid_gap="12px",
        width="100%"
    )
)

dashboard = widgets.VBox(
    [
        title,
        controls,
        summary_output,
        graph_grid
    ],
    layout=widgets.Layout(
        width="100%",
        overflow="visible"
    )
)

display(dashboard)

update_dashboard()